# Complexity-03c — Naviguer le Zoo exécutable

[Source Python](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) · [Index](README.md) · [03 — Hiérarchie en temps](Complexity-03-HierarchieTemporelle-Python.ipynb) · [03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) · [03d — ETH annoncée](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb)

**Positionnement.** Cet approfondissement de 03 et compagnon de 03b élargit le temps de calcul à une carte de classes. Le [Complexity Zoo](https://complexityzoo.net), fondé par Scott Aaronson, recense de nombreuses classes ; une liste de noms ne suffit pas à comprendre comment les ressources distinguent leurs problèmes. Ici, trois gestes donnent une prise sur la carte : parcourir un graphe et vérifier un chemin ; comparer deux familles SAT et leurs algorithmes ; jouer avec des oracles finis.

**Objectifs.** À la fin du carnet, vous pourrez : distinguer calcul et certificat, expliquer comment 2-SAT utilise les composantes fortement connexes et comment DPLL branche, donner le sens de L, NL, P, NP et PSPACE, et expliquer ce que des jeux finis d'oracles illustrent sans prouver les théorèmes généraux. **Prérequis** : graphes orientés, booléens et ordre de grandeur. **Durée pédagogique** : 45 minutes.

Le notebook Python versionné lié en tête est la source consultée pour cette transposition C# native. Ses références historiques, y compris Baker–Gill–Solovay, sont **rapportées par lui** ; les publications externes n'ont pas été lues pour ce port. On distingue **mesuré** (les sorties de ce carnet), **rapporté** (un résultat que le source attribue à la littérature) et **ouvert**. Aucune inclusion ou séparation de classes n'est démontrée par un nombre fini d'exemples.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;
using System.Globalization;

// Les sorties numériques et les coordonnées des figures gardent le point décimal entre soumissions.
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;
Console.WriteLine("Structures et générateur .NET prêts");


Structures et générateur .NET prêts


## 1. Atteignabilité : chercher et vérifier

Le source prend pour fil la chaîne rapportée $L\subseteq NL\subseteq P\subseteq NP\subseteq PSPACE$. Elle compare des classes définies par les ressources disponibles sur **toutes** les entrées, pas les performances de quelques programmes. Le calcul qui suit ne démontre donc aucune flèche ; il rend visibles certains mécanismes employés dans ces définitions.

Sur un graphe orienté, l'atteignabilité demande s'il existe un chemin de $s$ à $t$. Le DFS déterministe conserve une pile et un ensemble de sommets déjà vus : on mesure ici la **taille maximale de la pile**, qui est une partie de sa mémoire de travail, et non une mesure de toute sa mémoire. Le générateur donne quatre arcs par sommet aux premiers graphes, puis deux aux graphes de la comparaison avec le certificat. Les tailles 40, 100, 250 et 600 et la graine sont conservées ; `System.Random` et `random` Python ne produisent pas les mêmes graphes.

Le vérificateur reçoit un chemin sous forme de flux, vérifie chaque arête et ne conserve que sommet courant et compteur de pas, hors stockage du graphe et du certificat. Un producteur BFS distinct construit le témoin quand un chemin existe : son coût n'est pas effacé, il est simplement **hors du vérificateur**. Sur une entrée négative, aucun chemin n'est produit ; cela ne signifie pas qu'un témoin inexistant certifie directement la non-atteignabilité. Un chemin amputé de son arrivée fournit un contrôle négatif quand le producteur en a trouvé un.

In [2]:
var rng = new Random(20260925);
// Chaque sommet reçoit exactement le nombre d’arcs demandé ; les boucles et doublons restent possibles.
List<int>[] Graphe(int n, int degre)
{
    var a = Enumerable.Range(0, n).Select(_ => new List<int>()).ToArray();
    for (int u = 0; u < n; u++)
        for (int i = 0; i < degre; i++)
            a[u].Add(rng.Next(n));
    return a;
}

// DFS compte sa pile maximale ; le résultat ne comprend pas le coût du certificat externe.
(bool, int) Dfs(List<int>[] a, int source, int cible)
{
    var vus = new HashSet<int>
    {
        source
    };
    var pile = new Stack<int>();
    pile.Push(source);
    int max = 1;
    while (pile.Count > 0)
    {
        int u = pile.Pop();
        if (u == cible)
            return (true, max);
        foreach (int v in a[u])
            if (vus.Add(v))
            {
                pile.Push(v);
                max = Math.Max(max, pile.Count);
            }
    }

    return (false, max);
}

// Le producteur BFS reconstruit le témoin à rebours à partir des parents.
List<int> Chemin(List<int>[] a, int source, int cible)
{
    var file = new Queue<int>();
    var parent = Enumerable.Repeat(-1, a.Length).ToArray();
    file.Enqueue(source);
    parent[source] = source;
    while (file.Count > 0)
    {
        int u = file.Dequeue();
        if (u == cible)
        {
            var chemin = new List<int>();
            while (u != source)
            {
                chemin.Add(u);
                u = parent[u];
            }

            chemin.Reverse();
            return chemin;
        }

        foreach (int v in a[u])
            if (parent[v] < 0)
            {
                parent[v] = u;
                file.Enqueue(v);
            }
    }

    return null;
}

// Le vérificateur lit les sommets une fois et rejette une arête absente ou un chemin trop long.
bool VerifierChemin(List<int>[] a, int source, int cible, IEnumerable<int> certificat)
{
    int u = source, pas = 0;
    foreach (int v in certificat)
    {
        if (++pas > a.Length || v < 0 || v >= a.Length || !a[u].Contains(v))
            return false;
        u = v;
    }

    return u == cible;
}

foreach (int n in new[]
{
    40,
    100,
    250,
    600
}

)
{
    var a = Graphe(n, 4);
    var montre = System.Diagnostics.Stopwatch.StartNew();
    var r = Dfs(a, 0, n - 1);
    montre.Stop();
    Console.WriteLine($"DFS n={n} atteint={r.Item1}, pile max={r.Item2}, temps={montre.Elapsed.TotalMilliseconds:F3} ms");
}

foreach (int n in new[]
{
    40,
    100,
    250,
    600
}

)
{
    var a = Graphe(n, 2);
    var dfs = Dfs(a, 0, n - 1);
    var chemin = Chemin(a, 0, n - 1);
    bool verifie = chemin != null && VerifierChemin(a, 0, n - 1, chemin);
    Console.WriteLine($"n={n} DFS={dfs.Item1}, certificat={verifie}, accord={dfs.Item1 == verifie}, longueur={chemin?.Count ?? 0}");
    if (chemin != null && chemin.Count > 0)
        Console.WriteLine($"  témoin amputé refusé : {!VerifierChemin(a, 0, n - 1, chemin.Take(chemin.Count - 1))}");
}


DFS n=40 atteint=True, pile max=8, temps=0.463 ms


DFS n=100 atteint=True, pile max=40, temps=0.022 ms


DFS n=250 atteint=True, pile max=109, temps=0.024 ms


DFS n=600 atteint=True, pile max=262, temps=0.056 ms


n=40 DFS=True, certificat=True, accord=True, longueur=2


  témoin amputé refusé : True


n=100 DFS=True, certificat=True, accord=True, longueur=3


  témoin amputé refusé : True


n=250 DFS=True, certificat=True, accord=True, longueur=7


  témoin amputé refusé : True


n=600 DFS=False, certificat=False, accord=True, longueur=0


### Lecture — deux comptes de mémoire, deux rôles

La sortie donne pour chaque taille le verdict DFS, la pile maximale et son temps de mesure. Sur les secondes instances, le tableau confronte le verdict DFS à la lecture du chemin produit par BFS ; le dernier graphe peut être inatteignable selon les tirages. Lorsque le témoin existe, le chemin amputé doit être refusé. L'accord sur ces instances teste la cohérence des deux programmes ; le refus du faux témoin montre que la vérification ne consiste pas simplement à toujours accepter.

Deux mots de travail pour lire un témoin ne disent pas que son **producteur** n'a coûté que deux mots : BFS garde ses parents et sa file. Ils ne prouvent pas non plus $L=NL$, ni même l'inclusion de classes par l'expérience. Le nondéterminisme se comprend comme une recherche possible d'un témoin, pas comme l'effacement gratuit de la construction dans une exécution déterministe réelle. Les chronos sont propres à la machine et se lisent dans la sortie plutôt que dans une constante recopiée.

Sur les graphes à quatre arcs par sommet, les piles maximales observées sont **8, 40, 109 et 262** pour $n=40,100,250,600$. Sur la seconde famille, à deux arcs par sommet, les trois premiers exemples ont un chemin de longueurs 2, 3 et 7, et les témoins amputés sont refusés ; pour $n=600$, DFS et le producteur n'en trouvent pas, et les deux verdicts affichés sont `False`. Les deux familles sont distinctes : ne pas associer les piles de la première aux chemins de la seconde.

## Exercice 1 — Deux chemins contradictoires dans 2-SAT

Dans le graphe d'implications, une formule est insatisfiable lorsqu'une variable et sa négation sont mutuellement atteignables. Vérifier un certificat formé de deux chemins opposés ; un seul chemin ne suffit pas à décider une formule générale.

In [3]:
bool? CertificatContradiction(List<int>[] implications, int x, int nonX, int[] aller, int[] retour)
{
    // TODO : vérifier séparément les deux chemins orientés.
    return null;
}

var gEx = new[]
{
    new List<int>
    {
        1
    },
    new List<int>
    {
        0,
        2
    },
    new List<int>()
};
var ex1 = CertificatContradiction(gEx, 0, 1, new[] { 1 }, new[] { 0 });
if (ex1 == null)
    Console.WriteLine("Exercice 1 à compléter");
else
{
    if (ex1 != true || CertificatContradiction(gEx, 0, 1, new[] { 1 }, new[] { 2 }) != false || CertificatContradiction(gEx, 0, 1, new[] { 2 }, new[] { 0 }) != false)
        throw new Exception("Le certificat doit vérifier les deux chemins");
    Console.WriteLine("Deux chemins valides et deux témoins incorrects vérifiés");
}


Exercice 1 à compléter


## 2. P, NP et les deux familles SAT

Une clause $a\lor b$ de 2-SAT correspond aux implications $\neg a\Rightarrow b$ et $\neg b\Rightarrow a$. Une formule 2-CNF est insatisfiable exactement lorsqu'une variable et sa négation sont dans la même composante fortement connexe. Kosaraju parcourt le graphe d'implications, ordonne les sommets par fin de parcours puis parcourt le graphe transposé : son coût est linéaire en sommets et arêtes. Le programme produit aussi une affectation lorsque la formule est satisfiable.

Pour 3-SAT, DPLL tient une affectation partielle. Il propage les clauses unitaires, détecte une clause devenue vide, puis choisit un littéral et explore ses deux valeurs si nécessaire. Le nombre de nœuds donne une mesure déterministe pour une formule et une stratégie de branchement données ; il n'est pas une borne inférieure générale pour toutes les stratégies possibles. Une affectation renvoyée est vérifiée à part sur les clauses initiales : une recherche et le contrôle de son certificat ne sont pas le même geste.

Le banc suit les budgets du source : 2-SAT à 50, 200 et 800 variables, $3n$ clauses, cinq instances et trois chronos ; 3-SAT à 10, 15, 20, 25 et 30 variables, $4{,}6n$ clauses, vingt instances et trois chronos. Le minimum des trois temps réduit une partie du bruit de mesure ; la moyenne porte sur les instances. La densité voisine de la zone difficile des formules aléatoires est un choix d'expérience, pas une garantie que chaque formule soit difficile.

In [4]:
// Chaque clause (a ou b) fournit les implications non-a -> b et non-b -> a.
(bool, int[]) DeuxSat(int[][] clauses, int n)
{
    int Id(int l) => l > 0 ? l - 1 : -l - 1 + n;
    int Neg(int i) => i < n ? i + n : i - n;
    var a = Enumerable.Range(0, 2 * n).Select(_ => new List<int>()).ToArray();
    var b = Enumerable.Range(0, 2 * n).Select(_ => new List<int>()).ToArray();
    foreach (var c in clauses)
    {
        int x = Id(c[0]), y = Id(c[1]);
        a[Neg(x)].Add(y);
        a[Neg(y)].Add(x);
        b[y].Add(Neg(x));
        b[x].Add(Neg(y));
    }

    // Kosaraju : ordre de fin sur le graphe, puis composantes sur le graphe transposé.
    var vu = new bool[2 * n];
    var ordre = new List<int>();
    void Visite(int u)
    {
        vu[u] = true;
        foreach (int v in a[u])
            if (!vu[v])
                Visite(v);
        ordre.Add(u);
    }

    for (int i = 0; i < 2 * n; i++)
        if (!vu[i])
            Visite(i);
    var comp = Enumerable.Repeat(-1, 2 * n).ToArray();
    void Attribue(int u, int c)
    {
        comp[u] = c;
        foreach (int v in b[u])
            if (comp[v] < 0)
                Attribue(v, c);
    }

    int id = 0;
    ordre.Reverse();
    foreach (int u in ordre)
        if (comp[u] < 0)
            Attribue(u, id++);
    if (Enumerable.Range(0, n).Any(i => comp[i] == comp[i + n]))
        return (false, null);
    return (true, Enumerable.Range(0, n).Select(i => comp[i] > comp[i + n] ? 1 : 0).ToArray());
}

// DPLL compte chaque nœud ; une clause vide est une contradiction immédiate.
(bool, Dictionary<int, bool>, int) TroisSat(int[][] clauses)
{
    int noeuds = 0;
    Dictionary<int, bool> Resoudre(List<int[]> restantes, Dictionary<int, bool> partiel)
    {
        noeuds++;
        while (true)
        {
            if (restantes.Any(c => c.Length == 0))
                return null;
            if (restantes.Count == 0)
                return partiel;
            var unit = restantes.FirstOrDefault(c => c.Length == 1);
            if (unit == null)
                break;
            int l = unit[0];
            if (partiel.TryGetValue(Math.Abs(l), out bool ancien) && ancien != (l > 0))
                return null;
            partiel = new Dictionary<int, bool>(partiel);
            partiel[Math.Abs(l)] = l > 0;
            restantes = restantes.Where(c => !c.Contains(l)).Select(c => c.Where(x => x != -l).ToArray()).ToList();
        }

        // La propagation est épuisée : explorer les deux valeurs du littéral choisi.
        int choix = restantes[0][0];
        foreach (int l in new[]
        {
            choix,
            -choix
        }

        )
        {
            var copie = new Dictionary<int, bool>(partiel)
            {
                [Math.Abs(l)] = l > 0
            };
            var suite = restantes.Where(c => !c.Contains(l)).Select(c => c.Where(x => x != -l).ToArray()).ToList();
            var solution = Resoudre(suite, copie);
            if (solution != null)
                return solution;
        }

        return null;
    }

    var solution = Resoudre(clauses.ToList(), new Dictionary<int, bool>());
    return (solution != null, solution, noeuds);
}

bool Verifier3(int[][] clauses, Dictionary<int, bool> solution) => clauses.All(c => c.Any(l => solution.TryGetValue(Math.Abs(l), out bool v) && v == (l > 0)));
int[][] Generer(int n, int m, int largeur)
{
    var clauses = new int[m][];
    for (int i = 0; i < m; i++)
    {
        var vars = largeur == 2 ? new[]
        {
            rng.Next(1, n + 1),
            rng.Next(1, n + 1)
        }

        : Enumerable.Range(1, n).OrderBy(_ => rng.Next()).Take(largeur).ToArray();
        clauses[i] = vars.Select(x => rng.Next(2) == 0 ? x : -x).ToArray();
    }

    return clauses;
}

Console.WriteLine("Solveurs 2-SAT et DPLL définis ; une clause vide est explicitement insatisfiable");


Solveurs 2-SAT et DPLL définis ; une clause vide est explicitement insatisfiable


### Lecture de l'implémentation

Les deux parcours de composantes utilisent les graphes direct et transposé ; si les composantes d'un littéral et de sa négation coïncident, aucune affectation ne peut satisfaire les implications. DPLL simplifie chaque clause après une affectation, conserve une clause vide comme contradiction et ne déclare le succès que lorsque toutes les clauses ont été satisfaites. La vérification de certificat parcourt de nouveau la formule d'origine. Ces contrôles portent sur les résultats rendus, pas seulement sur l'absence d'exception.

In [5]:
// Même protocole que la source : trois chronos par instance et moyenne des minima.
var montres2 = new Dictionary<int, double>();
foreach (int n in new[]
{
    50,
    200,
    800
}

)
{
    double cumul = 0;
    for (int essai = 0; essai < 5; essai++)
    {
        var f = Generer(n, 3 * n, 2);
        double meilleur = double.PositiveInfinity;
        for (int rep = 0; rep < 3; rep++)
        {
            var sw = System.Diagnostics.Stopwatch.StartNew();
            DeuxSat(f, n);
            sw.Stop();
            meilleur = Math.Min(meilleur, sw.Elapsed.TotalMilliseconds);
        }

        cumul += meilleur;
    }

    montres2[n] = cumul / 5;
    Console.WriteLine($"2-SAT n={n}, moyenne des minima (5 instances × 3) : {cumul / 5:F3} ms");
}

// La densité 4,6n et le nombre d’instances sont conservés pour chaque taille.
var noeudsMoyens = new Dictionary<int, double>();
var tempsMoyens = new Dictionary<int, double>();
foreach (int n in new[]
{
    10,
    15,
    20,
    25,
    30
}

)
{
    double temps = 0, noeuds = 0;
    for (int essai = 0; essai < 20; essai++)
    {
        var f = Generer(n, (int)(4.6 * n), 3);
        double meilleur = double.PositiveInfinity;
        int count = 0;
        for (int rep = 0; rep < 3; rep++)
        {
            var sw = System.Diagnostics.Stopwatch.StartNew();
            var sol = TroisSat(f);
            sw.Stop();
            meilleur = Math.Min(meilleur, sw.Elapsed.TotalMilliseconds);
            count = sol.Item3;
        }

        temps += meilleur;
        noeuds += count;
    }

    noeudsMoyens[n] = noeuds / 20;
    tempsMoyens[n] = temps / 20;
    Console.WriteLine($"3-SAT n={n}, 20 instances × 3 : noeuds moyens={noeuds / 20:F1}, temps moyen={temps / 20:F3} ms");
}

// La solution fournie par DPLL est contrôlée sur la formule initiale.
int certificats = 0;
for (int essai = 0; essai < 20; essai++)
{
    int[][] f = null;
    (bool, Dictionary<int, bool>, int) sol = default;
    for (int tentative = 0; tentative < 12; tentative++)
    {
        f = Generer(12, 48, 3);
        sol = TroisSat(f);
        if (sol.Item1)
            break;
    }

    if (sol.Item1 && Verifier3(f, sol.Item2))
        certificats++;
}

double facteurNoeuds = Math.Pow(noeudsMoyens[30] / noeudsMoyens[10], 1.0 / 20);
double facteurTemps = Math.Pow(tempsMoyens[30] / tempsMoyens[10], 1.0 / 20);
Console.WriteLine($"Certificats vérifiés {certificats}/20 ; facteur estimé par variable : noeuds={facteurNoeuds:F3}, temps={facteurTemps:F3}");


2-SAT n=50, moyenne des minima (5 instances × 3) : 0.042 ms


2-SAT n=200, moyenne des minima (5 instances × 3) : 0.152 ms


2-SAT n=800, moyenne des minima (5 instances × 3) : 0.560 ms


3-SAT n=10, 20 instances × 3 : noeuds moyens=9.2, temps moyen=0.372 ms


3-SAT n=15, 20 instances × 3 : noeuds moyens=14.4, temps moyen=1.134 ms


3-SAT n=20, 20 instances × 3 : noeuds moyens=25.8, temps moyen=3.302 ms


3-SAT n=25, 20 instances × 3 : noeuds moyens=37.2, temps moyen=5.895 ms


3-SAT n=30, 20 instances × 3 : noeuds moyens=57.8, temps moyen=9.870 ms


Certificats vérifiés 20/20 ; facteur estimé par variable : noeuds=1.097, temps=1.178


### Lecture — ce que la pente mesure

La sortie donne pour 2-SAT la moyenne des minima de temps, et pour 3-SAT à la fois les nœuds moyens et les temps. Les tailles comparées ne sont pas les mêmes : dire que l'un est « plus rapide » à partir d'un seul temps brut mélangerait la taille, l'instance et l'implémentation. Le facteur par variable calculé entre les extrémités est une description de **ce banc**, non une loi universelle de croissance. Il peut changer avec les graines, la densité ou le choix du branchement.

Les vingt tentatives de certificats partent de formules candidates à 12 variables et 48 clauses, avec jusqu'à douze essais pour trouver une formule satisfiable ; le nombre effectivement vérifié est imprimé. Vérifier qu'une affectation rend chaque clause vraie prend un parcours de la formule. Cela illustre la distinction entre chercher et vérifier, sans résoudre la question $P$ versus $NP$.

Sur ce tirage seedé, les nœuds moyens DPLL passent de **9,2** à $n=10$ à **57,8** à $n=30$, soit un facteur par variable ajouté d'environ **1,097** calculé entre ces deux points. Les **20/20** certificats présentés sont vérifiés. La valeur numérique du facteur de temps dépend de l'environnement ; elle reste dans la sortie. Ni ces points ni le facteur ne constituent une preuve d'une exponentielle pour tous les cas.

## Exercice 2 — Projeter les temps DPLL

À partir du dernier temps mesuré et du facteur par variable, projeter n=40 et n=60 puis calculer le premier n dépassant une heure. Si le facteur vaut au plus 1, indiquer qu'aucun dépassement ne découle de cette extrapolation.

In [6]:
(double, double, int? )? Projection(double temps30Secondes, double facteur)
{
    // TODO : projeter et ne renvoyer une taille finie que si le facteur le permet.
    return null;
}

var ex2 = Projection(1.0, 2.0);
if (ex2 == null)
    Console.WriteLine("Exercice 2 à compléter");
else
{
    var p = ex2.Value;
    if (Math.Abs(p.Item1 - 1024) > 1e-8 || Math.Abs(p.Item2 - Math.Pow(2, 30)) > 1e-3 || p.Item3 != 42 || Projection(1, 1).Value.Item3 != null)
        throw new Exception("Projection incorrecte");
    Console.WriteLine("Projection et absence de dépassement si facteur 1 vérifiées");
    var reel = Projection(tempsMoyens[30] / 1000.0, facteurTemps);
    if (reel == null)
        throw new Exception("Projection absente sur les mesures");
    double baseReelle = tempsMoyens[30] / 1000.0;
    if (Math.Abs(reel.Value.Item1 - baseReelle * Math.Pow(facteurTemps, 10)) > 1e-8 || Math.Abs(reel.Value.Item2 - baseReelle * Math.Pow(facteurTemps, 30)) > 1e-8)
        throw new Exception("Projection des mesures incorrecte");
    int? seuilAttendu = null;
    if (baseReelle > 3600)
        seuilAttendu = 30;
    else if (facteurTemps > 1)
    {
        int n = 30;
        double temps = baseReelle;
        while (temps <= 3600 && n < 10000)
        {
            temps *= facteurTemps;
            n++;
        }

        if (temps > 3600)
            seuilAttendu = n;
    }

    if (reel.Value.Item3 != seuilAttendu)
        throw new Exception("Seuil d'une heure incorrect sur les mesures");
    Console.WriteLine($"Mesures réelles : n=40 {reel.Value.Item1:F4}s, n=60 {reel.Value.Item2:F4}s, seuil 1h n={reel.Value.Item3?.ToString() ?? "non atteint"}");
}


Exercice 2 à compléter


## 3. Deux jeux d'oracles finis et la relativisation

Le notebook Python rapporte le théorème de Baker, Gill et Solovay (1975) : il existe des oracles infinis $A$ et $B$ tels que $P^A\ne NP^A$ et $P^B=NP^B$. Une preuve qui se relativise reste valable lorsqu'on donne le même oracle aux deux côtés ; elle ne peut donc, à elle seule, décider $P$ contre $NP$ dans le monde sans oracle. Certaines diagonalisations se relativisent : le théorème ne disqualifie pas toute technique portant le nom de diagonalisation. Ce résultat historique est **rapporté**, non établi par nos programmes.

Le jeu A retient seulement quatre machines déterministes particulières. À un stade de longueur fraîche $n$, chacune examine au plus $n$ candidats parmi $2^n$ ; le programme choisit une réponse par défaut puis, si nécessaire, place un témoin hors des positions examinées. Un second calcul énumère tous les témoins de cette entrée finie. Un désaccord est observable pour chaque machine du banc. La logique « longueur fraîche, témoin caché » rappelle une partie du mécanisme, sans quantifier sur toutes les machines ni construire un oracle infini.

Le jeu B délègue à une fonction-oracle une recherche existentielle sur cinq entrées de cinq bits ; le coût interne de cette fonction est omis du nombre de requêtes de l'appelant, mais reste bien réel pour notre simulation. L'exemple hérité répond oui sur chacune de ces cinq entrées. Un second prédicat, ajouté pour rendre la différence observable, rend un témoin pour une entrée et aucun pour une autre. Aucune de ces observations ne prouve $P^B=NP^B$.

In [7]:
int MachineP(HashSet<string> oracle, string x, int defaut)
{
    int n = x.Length, positives = 0;
    for (int y = 0; y < n; y++)
        if (oracle.Contains(x + "#" + Convert.ToString(y, 2).PadLeft(n, '0')))
            positives++;
    return positives == 0 ? defaut : (positives > n / 2 ? 1 : 0);
}

int Langage(HashSet<string> oracle, string x)
{
    int n = x.Length;
    for (int y = 0; y < (1 << n); y++)
        if (oracle.Contains(x + "#" + Convert.ToString(y, 2).PadLeft(n, '0')))
            return 1;
    return 0;
}

var oracle = new HashSet<string>();
var stades = new List<(int i, int n, int reponse, int reel)>();
for (int i = 1; i <= 4; i++)
{
    int n = 6 + 2 * i;
    string x = Convert.ToString(i, 2).PadLeft(n, '0');
    int rep = MachineP(oracle, x, i % 2);
    if (rep == 0)
    {
        int y = (i * 7919) % (1 << n);
        if (y < n)
            y = n;
        oracle.Add(x + "#" + Convert.ToString(y, 2).PadLeft(n, '0'));
    }

    int reel = Langage(oracle, x);
    stades.Add((i, n, rep, reel));
    Console.WriteLine($"M{i}, n={n}, réponse={rep}, témoin exhaustif={reel}, contredite={rep != reel}");
}

bool Predicat(string x, string y) => x.Count(c => c == '1') + y.Count(c => c == '1') >= x.Length;
int OracleExistentiel(string x)
{
    for (int y = 0; y < (1 << x.Length); y++)
        if (Predicat(x, Convert.ToString(y, 2).PadLeft(x.Length, '0')))
            return 1;
    return 0;
}

foreach (int i in new[]
{
    0,
    5,
    13,
    21,
    31
}

)
{
    string x = Convert.ToString(i, 2).PadLeft(5, '0');
    int reponse = OracleExistentiel(x);
    int exhaustif = Enumerable.Range(0, 32).Any(y => Predicat(x, Convert.ToString(y, 2).PadLeft(5, '0'))) ? 1 : 0;
    Console.WriteLine($"x={x}, oracle une requête={reponse}, recherche exhaustive={exhaustif}");
}

Console.WriteLine("Quatre stades finis et cinq entrées : illustrations seulement");
// Un second prédicat fournit une réponse positive et une négative.
bool PredicatMixte(string x, string y) => x.All(c => c == '0') && y.All(c => c == '0');
foreach (string x in new[]
{
    "00000",
    "00101"
}

)
{
    string temoin = Enumerable.Range(0, 32).Select(y => Convert.ToString(y, 2).PadLeft(5, '0')).FirstOrDefault(y => PredicatMixte(x, y));
    bool existe = temoin != null;
    if (existe && !PredicatMixte(x, temoin))
        throw new Exception("Témoin incohérent");
    Console.WriteLine($"Prédicat mixte x={x} : existe={existe}, témoin={temoin ?? "aucun"}");
}


M1, n=8, réponse=1, témoin exhaustif=0, contredite=True


M2, n=10, réponse=0, témoin exhaustif=1, contredite=True


M3, n=12, réponse=1, témoin exhaustif=0, contredite=True


M4, n=14, réponse=0, témoin exhaustif=1, contredite=True


x=00000, oracle une requête=1, recherche exhaustive=1


x=00101, oracle une requête=1, recherche exhaustive=1


... (2 outputs left out)

x=11111, oracle une requête=1, recherche exhaustive=1


Quatre stades finis et cinq entrées : illustrations seulement


Prédicat mixte x=00000 : existe=True, témoin=00000


Prédicat mixte x=00101 : existe=False, témoin=aucun


### Lecture — un témoin fini ne remplace pas un théorème

Les quatre lignes du premier jeu montrent la réponse de la machine, celle de l'énumération exhaustive et le désaccord. Les cinq lignes du second comparent une requête déléguée et une recherche explicite du même prédicat ; ici, le témoin tout à un permet toujours un oui. Les deux lignes supplémentaires montrent aussi le cas négatif et le témoin positif vérifié. Le nombre « une requête » mesure l'interface de l'oracle, non le travail que la simulation a effectué à l'intérieur.

La relativisation explique la portée d'une **famille de méthodes de preuve**. Les jeux gardent le contraste pédagogique « accès à une boîte noire », mais ni leurs cinq entrées ni leurs quatre machines ne suffisent à construire les mondes mathématiques infinis de l'énoncé cité.

## Exercice 3 — Ajouter un stade

Étendre le premier jeu pour une cinquième machine à une longueur fraîche. Choisir un témoin hors des n premières requêtes si nécessaire et vérifier que les quatre résultats précédents restent inchangés.

In [8]:
(HashSet<string>, List<(int, int, int, int)>)? EtendreOracle(HashSet<string> courant, int indice)
{
    // TODO : copier l'oracle, choisir la longueur fraîche et vérifier le nouveau stade.
    return null;
}

var ex3 = EtendreOracle(oracle, 5);
if (ex3 == null)
    Console.WriteLine("Exercice 3 à compléter");
else
{
    var(nouveau, historique) = ex3.Value;
    int n = 16;
    string x = Convert.ToString(5, 2).PadLeft(n, '0');
    if (nouveau == null || historique == null || !historique.Any(s => s.Item1 == 5 && s.Item2 == 16) || MachineP(nouveau, x, 1) == Langage(nouveau, x) || stades.Any(s => Langage(nouveau, Convert.ToString(s.i, 2).PadLeft(s.n, '0')) != s.reel))
        throw new Exception("Le stade ne contredit pas la machine ou modifie les anciens");
    Console.WriteLine("Cinquième stade et anciens résultats vérifiés");
}


Exercice 3 à compléter


## 4. Carte du Zoo et limites

La carte distingue des ressources et des modes de calcul. **L** regroupe les problèmes décidables en espace logarithmique déterministe ; **NL** autorise le nondéterminisme avec le même ordre d'espace ; **P** regroupe les problèmes décidables en temps polynomial déterministe ; **NP** ceux dont un certificat polynomial se vérifie en temps polynomial ; **coNP** contient les complémentaires des problèmes de NP. **PSPACE** autorise un espace polynomial, et **EXPTIME** un temps exponentiel.

| Relation | Statut rapporté par le source Python | Illustration ici |
|---|---|---|
| L ⊆ NL | Inclusion établie | certificat de chemin lu une fois ; le BFS qui le construit est hors du vérificateur |
| NL ⊆ P | Inclusion établie | atteignabilité décidée par parcours déterministe |
| P ⊆ NP | Inclusion établie | vérification de certificat 3-SAT |
| NP ⊆ PSPACE | Inclusion établie | citée, non démontrée par ce carnet |
| P ⊊ EXPTIME | Hiérarchie en temps | citée, non démontrée par ce carnet |
| P vs NP | Ouvert dans le source | jeux d'oracles finis, pas une preuve générale |
| NP vs coNP | Ouvert dans le source | la non-satisfiabilité ne dispose pas d'un certificat polynomial général connu |
| NP vs PSPACE | Ouvert dans le source | l'égalité IP = PSPACE n'établit pas cette séparation |

La barrière de relativisation est rapportée par le source : l'existence de deux oracles infinis de comportements opposés limite les preuves qui se relativisent. Les jeux finis de §3 n'établissent ni ces oracles infinis ni une séparation de classes. L'encart Mathlib du source (constat historique) distingue la calculabilité formalisée des classes de complexité bornées, sans en déduire une absence actuelle exhaustive.

```mermaid
flowchart LR
    L[L] --> NL[NL] --> P[P] --> NP[NP] --> PS[PSPACE]
    P --> EX[EXPTIME]
    P -. ouvert .- NP
    NP -. ouvert .- coNP[coNP]
    NP -. ouvert .- PS
```

Les traits pleins sont les inclusions rapportées ; $P\subsetneq EXPTIME$ est la séparation stricte attribuée à la hiérarchie en temps. Les pointillés indiquent des questions ouvertes dans le source, pas des relations prouvées. L'absence d'un certificat polynomial général connu pour la non-satisfiabilité est une formulation de la difficulté autour de NP/coNP, pas une preuve d'impossibilité.

### Encart Mathlib — portée historique du relevé Python

Le source Python décrivait une infrastructure `Computability/` pour les machines de Turing, fonctions partielles et réductions. La calculabilité non bornée constitue un socle de définition, **pas la classe L** : pour parler de L, il faut aussi une borne de mémoire. Le relevé du source ne trouvait pas les couches de complexité bornée et d'oracles nécessaires pour formaliser son Zoo ; ce n'est pas un nouvel audit exhaustif de la version actuelle de Mathlib. Ici, les algorithmes sont exécutables et les relations de classes restent rapportées, pas formellement prouvées.

## Conclusion — une carte, trois niveaux de preuve

| Geste | Résultat disponible ici | Limite |
|---|---|---|
| Recherche et certificat | DFS, BFS producteur et vérification d'un chemin ; contrôle d'un faux témoin | Le coût de production du témoin reste distinct du coût de lecture |
| SAT | Kosaraju 2-SAT, DPLL 3-SAT, mesures et vérification de certificats | Des essais finis n'établissent pas une borne de complexité générale |
| Oracles | Quatre stades finis et deux prédicats existentiels | Ce ne sont pas les oracles infinis du théorème BGS |
| Carte et Mathlib | Inclusions rapportées, questions ouvertes et relevé historique du source | Aucune preuve formelle de ces classes n'est produite ici |

## À retenir

1. **Chercher et vérifier ont des entrées différentes.** Le chemin est fourni au vérificateur ; BFS le construit dans la démonstration, avec sa propre mémoire.
2. **La pile DFS est une mesure partielle de mémoire.** La sortie indique sa taille maximale sur chaque graphe, sans compter l'ensemble des sommets visités.
3. **2-SAT et 3-SAT sont étudiés sur des familles distinctes.** La première utilise des composantes fortement connexes ; la seconde explore des branchements et son compteur concerne la stratégie choisie.
4. **Les oracles du banc sont finis.** Quatre désaccords et quelques réponses existentielles illustrent des mécanismes mais ne prouvent pas de séparation de classes.
5. **La carte sépare résultat rapporté et question ouverte.** La relativisation limite les techniques qui se relativisent ; le carnet ne démontre pas la barrière par son jeu.

## Vérifiez votre compréhension

1. Quel rôle jouent l'accord DFS/certificat et le refus d'un faux témoin ?
2. Pourquoi le facteur de croissance en nœuds de DPLL est-il plus informatif qu'une comparaison brute de temps entre 2-SAT à 800 variables et 3-SAT à 30 ?
3. Que prouvent les quatre désaccords du jeu A, et que ne prouvent-ils pas ?
4. Pourquoi distinguer une inclusion rapportée d'un résultat observé sur une instance ?

<details><summary>Éléments de réponse</summary>

1. Ils contrôlent la cohérence sur les cas étudiés et rejettent un témoin particulier invalide ; ils ne prouvent pas une inclusion de classes.
2. Il décrit la croissance mesurée pour une même stratégie, même si les graines et la taille continuent d'influencer le résultat ; les temps bruts mélangent implémentations et tailles.
3. Ils démontrent les désaccords de quatre machines et entrées finies, pas une séparation pour toutes les machines à oracle.
4. Une observation finie peut servir d'exemple ou de contre-exemple, mais une inclusion quantifie sur une classe entière et ses entrées.
</details>

## Pour aller plus loin

[Source Python](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) · [03b — hiérarchie](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) · [03d — ETH annoncée](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) · [04 — décisions en ligne](Complexity-04-OnlineAlgorithms-Python.ipynb) · [05 — compter](Complexity-05-CountingHarder-Permanent-Python.ipynb).

**Références rapportées par le source Python** : Baker, Gill et Solovay, *Relativizations of the P=?NP Question* (1975), pour la barrière ; *The CP-SAT Primer* (2023), pour le contexte de solveurs SAT. Ces publications n'ont pas été consultées pour ce port ; les mesures et le code exécuté sont ceux de ce carnet.